# Chapter 5 MyElite fine-tuning with Qwen

- 데이터셋: `/data/chapter5/loyalty_qa_train.jsonl`, `/data/chapter5/loyalty_qa_val.jsonl`
- 모델: `Qwen/Qwen2.5-0.5B-Instruct`
- 산출물: `/model-assets`


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("torch") is None:
  raise RuntimeError("PyTorch is missing. Check the JupyterHub image tag in eks/values/jupyterhub/values-example.yaml.")

package_modules = {
  "transformers": "transformers",
  "datasets": "datasets",
  "peft": "peft",
  "accelerate": "accelerate",
  "sentencepiece": "sentencepiece",
  "protobuf": "google.protobuf",
}

missing_packages = [
  package
  for package, module in package_modules.items()
  if importlib.util.find_spec(module) is None
]

if missing_packages:
  subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

print("missing packages installed:", missing_packages or "none")


In [ ]:
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
TRAIN_DATASET_FILE = "/data/chapter5/loyalty_qa_train.jsonl"
EVAL_DATASET_FILE = "/data/chapter5/loyalty_qa_val.jsonl"
MODEL_OUTPUT_DIR = Path("/model-assets")
MAX_STEPS = 20
MAX_LENGTH = 512

print("cuda available:", torch.cuda.is_available())
print("train dataset:", TRAIN_DATASET_FILE)
print("eval dataset:", EVAL_DATASET_FILE)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

model_options = {"torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32, "trust_remote_code": True}
if torch.cuda.is_available():
  model_options["device_map"] = "auto"

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **model_options)
model.config.use_cache = False
model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
  model.enable_input_require_grads()

config = LoraConfig(
  r=16,
  lora_alpha=32,
  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
  bias="none",
  lora_dropout=0.05,
  task_type="CAUSAL_LM",
)
model = get_peft_model(model, config)
model.print_trainable_parameters()


In [ ]:
def format_training_text(example):
  messages = [
    {"role": "user", "content": example["prompt"]},
    {"role": "assistant", "content": example["response"]},
  ]
  return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def tokenize_dataset(dataset):
  def tokenize_example(example):
    tokens = tokenizer(format_training_text(example), truncation=True, max_length=MAX_LENGTH)
    return tokens

  return dataset.map(tokenize_example, remove_columns=dataset.column_names)


train_dataset = tokenize_dataset(load_dataset("json", data_files=TRAIN_DATASET_FILE, split="train"))
eval_dataset = tokenize_dataset(load_dataset("json", data_files=EVAL_DATASET_FILE, split="train"))
len(train_dataset), len(eval_dataset)


In [ ]:
def move_inputs_to_device(inputs, device):
  if isinstance(inputs, torch.Tensor):
    return inputs.to(device)
  return {key: value.to(device) for key, value in inputs.items()}


def generate_with_inputs(inputs, **kwargs):
  if isinstance(inputs, torch.Tensor):
    return model.generate(inputs, **kwargs)
  return model.generate(**inputs, **kwargs)


def get_input_length(inputs):
  if isinstance(inputs, torch.Tensor):
    return inputs.shape[-1]
  return inputs["input_ids"].shape[-1]


def generate_text(prompt):
  messages = [{"role": "user", "content": prompt}]
  inputs = move_inputs_to_device(
    tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"),
    model.device,
  )
  with torch.no_grad():
    outputs = generate_with_inputs(inputs, max_new_tokens=100, repetition_penalty=1.15)
  return tokenizer.decode(outputs[0][get_input_length(inputs):], skip_special_tokens=True)


eval_prompt = "[MyElite Loyalty Program FAQ]: What is the maximum cashback I can earn?"
print(generate_text(eval_prompt))


In [ ]:
trainer = Trainer(
  model=model,
  train_dataset=train_dataset,
  eval_dataset=eval_dataset,
  args=TrainingArguments(
    output_dir="/tmp/qwen25-myelite-notebook",
    warmup_steps=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    max_steps=MAX_STEPS,
    learning_rate=2.5e-5,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    save_strategy="no",
    eval_strategy="steps",
    eval_steps=10,
    do_eval=True,
    report_to="none",
  ),
  data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()


In [ ]:
print(generate_text(eval_prompt))

MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))
print("saved:", MODEL_OUTPUT_DIR)
